# Notebook: PCR

## What this notebook does

This notebook is a **short, PCR-only extraction** from your 3G workflow.  
It keeps the original **PCR simulation functions from Plasmidio** (exact primers and 5' overhang mode) and adds a simple **TU (Transcription Unit) batch runner** so you can simulate PCR across many designs.

**What you can do here:**
- Use `simulate_pcr` (exact matches) or `simulate_pcr_overhangs` (longest unique anneal; 5' overhangs supported)
- Run PCR for a table of **TUs** (template + primers) and export results
- Preserve/shift **features** on products
- Save GenBank files for downstream analysis




In [2]:
# ------------------------------ Imports & setup

from __future__ import annotations

from pathlib import Path
import os  # optional; keep if you use e.g. os.environ / os.listdir
import re
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd

from Bio import SeqIO, pairwise2
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation, CompoundLocation

# Project API (top-level re-exports)
from assembly_designer import (
    # PCR & Gibson
    PCRResult,
    simulate_pcr,
    simulate_pcr_overhangs,
)

# Project API (submodule-only: not re-exported at top level)
from assembly_designer.plasmidio import (
    load_dna_file,
    remove_near_duplicate_features,
    _safe_filename,
)

# Optional dependency sanity check:
# - dnacauldron: required to simulate assemblies & write *_report.zip
# - snapgene_reader: enables reading .dna (SnapGene) files
try:
    import dnacauldron  # noqa: F401
    from snapgene_reader import snapgene_file_to_dict  # noqa: F401
    print("Optional deps OK: dnacauldron, snapgene-reader")
except Exception as err:
    print("⚠️ Optional deps check:", err)



Optional deps OK: dnacauldron, snapgene-reader


In [3]:
# ------------------------------ small helpers 
def _as_bool(x) -> bool:
    if isinstance(x, bool):
        return x
    if x is None:
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y", "ja"}

def _get_int(x, default: int) -> int:
    try:
        v = int(x)
        return v if v > 0 else default
    except Exception:
        return default
    
def _prefer(paths: List[Path], token: str) -> Path:
    for p in paths:
        if token.lower() in p.stem.lower():
            return p
    return paths[0]


## PCR 

In [4]:
# ------------------------------ Paths & loading 

BASE_DIR = Path.cwd()
ASSEMBLY_DIR = BASE_DIR / "reports" / "Assembly"
assert ASSEMBLY_DIR.is_dir(), f"Assembly folder not found: {ASSEMBLY_DIR}"

# Load two TU constructs (deterministic pick by UNS pair if available)
tu_paths: List[Path] = sorted(ASSEMBLY_DIR.glob("*.gb*"))
assert len(tu_paths) >= 2, f"Expected at least 2 TU .gb/.gbk files in {ASSEMBLY_DIR}"

In [5]:
# ------------------------------ Load all .gb/.gbk in the assembly folder (there should be exactly two final TUs)
tu_records: Dict[str, SeqRecord] = {}
for p in sorted(ASSEMBLY_DIR.glob("*.gb*")):
    rec = SeqIO.read(str(p), "genbank")
    stem = p.stem
    rec.id = stem
    rec.name = stem
    rec.annotations.setdefault("molecule_type", "DNA")
    tu_records[stem] = rec

print("Found TU constructs:", list(tu_records.keys()))
assert len(tu_records) >= 2, "Expected 2 TU constructs in the Assembly folder."

Found TU constructs: ['P45_AB_(A)_RBS_01_BC_(A)_E0030_CD_TrrnB_DE_(A)_UNS1A_UNS3_E', 'P45_AB_(A)_RBS_01_BC_(A)_E0040m_CD_TrrnB_DE_(A)_UNS3A_UNS10E', 'P69_AB_(A)_RBS_01_BC_(A)_E0030_CD_TrrnB_DE_(A)_UNS1A_UNS3_E', 'P69_AB_(A)_RBS_01_BC_(A)_E0040m_CD_TrrnB_DE_(A)_UNS3A_UNS10E']


In [6]:
# ------------------------------ Load data (no biology-specific logic)
BASE_DIR = Path.cwd()
ASSEMBLY_DIR = BASE_DIR / "reports" / "Assembly"
if not ASSEMBLY_DIR.is_dir():
    raise FileNotFoundError(f"Assembly folder not found: {ASSEMBLY_DIR}")

tu_paths: List[Path] = sorted(ASSEMBLY_DIR.glob("*.gb*"))
if len(tu_paths) < 2:
    raise FileNotFoundError(f"Expected at least 2 TU .gb/.gbk files in {ASSEMBLY_DIR}")

tu1_path = _prefer(tu_paths, "UNS1A_UNS3_E")
tu2_path = _prefer(tu_paths, "UNS3A_UNS10E")

tu1_rec: SeqRecord = SeqIO.read(str(tu1_path), "genbank")
tu2_rec: SeqRecord = SeqIO.read(str(tu2_path), "genbank")
for rec, p in [(tu1_rec, tu1_path), (tu2_rec, tu2_path)]:
    rec.id = rec.name = p.stem
    rec.annotations.setdefault("molecule_type", "DNA")

print("TUs:", tu1_rec.id, "|", tu2_rec.id)


TUs: P45_AB_(A)_RBS_01_BC_(A)_E0030_CD_TrrnB_DE_(A)_UNS1A_UNS3_E | P45_AB_(A)_RBS_01_BC_(A)_E0040m_CD_TrrnB_DE_(A)_UNS3A_UNS10E


In [8]:
# ------------------------------ Search for backboene vector file (prefer .dna if available)
CANDIDATES = [BASE_DIR / "Gibson_Backbone_parts",
              BASE_DIR / "Backbone_Parts",
              BASE_DIR / "Backbone_parts"]
vec_dir = next((d for d in CANDIDATES if d.is_dir()), None)

vec_files = [p for p in vec_dir.iterdir() if p.suffix.lower() in {".dna", ".gb", ".gbk", ".genbank"}]
if not vec_files:
    raise FileNotFoundError(f"No vector files in {vec_dir}")

vector_path = next((p for p in vec_files if p.suffix.lower() == ".dna"), vec_files[0])
vector_rec = load_dna_file(vector_path)

vector_rec.id = vector_rec.name = vector_path.stem
vector_rec.annotations.setdefault("molecule_type", "DNA")
print("Vector:", vector_rec.id)

Vector: pEVmC(K)_AE


In [10]:
# ------------------------------ Load Excel "gibson_designs_template.xlsx" and extract relevant sheets
xlsx_path = "PCR.xlsx" 
xl = pd.read_excel(xlsx_path, sheet_name=None)

constructs_df = xl.get("Constructs", pd.DataFrame())
pcr_df        = xl["PCR"].copy()

# vector_rec, tu1_rec, tu2_rec
role_records: Dict[str, SeqRecord] = {
    "Backbone": vector_rec,
    "TU1": tu1_rec,
    "TU2": tu2_rec,
}


In [11]:
len(tu1_rec)

1150

In [12]:
# ------------------------------ Generate all PCR products (Backbone supports overhangs) ---
products: Dict[Tuple[str, str], SeqRecord] = {}
pcr_results: Dict[Tuple[str, str], PCRResult] = {}  # keep full results if you want

for _, row in pcr_df.iterrows():
    cid  = str(row["ConstructID"])
    role = str(row["Role"])
    fwd  = str(row["FwdPrimer"]).replace(" ", "")
    rev  = str(row["RevPrimer"]).replace(" ", "")
    circ = _as_bool(row.get("Circular", True))
    ovh  = _as_bool(row.get("IncludeOverhangs", role.upper() == "BACKBONE"))
    min_anneal = _get_int(row.get("MinAnneal", 18), 18)

    template = role_records.get(role) or role_records.get(role.upper())
    if template is None:
        raise KeyError(f"No sequence record mapped for role '{role}'.")

    pid = f"{cid}_{role}_PCR"

    if ovh:
        res = simulate_pcr_overhangs(
            template,
            fwd_primer=fwd, rev_primer=rev,
            circular=circ, min_anneal=min_anneal,
            include_overhangs=True, product_id=pid,
        )
    else:
        res = simulate_pcr(
            template,
            fwd_primer=fwd, rev_primer=rev,
            circular=circ,
        )
        res.product.id = res.product.name = pid

    product = res.product

    # Attach PCR meta so history/plots can read primers directly from the product
    product.annotations["pcr_meta"] = {
        "construct_id": cid,
        "role": role,
        "fwd_primer": fwd,
        "rev_primer": rev,
        "circular": bool(circ),
        "include_overhangs": bool(ovh),
        "min_anneal": int(min_anneal),
        "fwd_anneal_len": getattr(res, "fwd_anneal_len", None),
        "rev_anneal_len": getattr(res, "rev_anneal_len", None),
    }

    products[(cid, role)] = product
    pcr_results[(cid, role)] = res  # optional but useful

print(f"PCR done for {len(products)} (ConstructID, Role) pairs.")


PCR done for 6 (ConstructID, Role) pairs.


In [13]:
products

{('C1',
  'Backbone'): SeqRecord(seq=Seq('CCAGGATACATAGATTACCACAACTCCGAGCCCTTCCACctactagtagcggcc...CTC'), id='C1_Backbone_PCR', name='C1_Backbone_PCR', description="PCR product with 5' overhangs (features preserved)", dbxrefs=[]),
 ('C2',
  'Backbone'): SeqRecord(seq=Seq('CCAGGATACATAGATTACCACAACTCCGAGCCCTTCCACctactagtagcggcc...CTC'), id='C2_Backbone_PCR', name='C2_Backbone_PCR', description="PCR product with 5' overhangs (features preserved)", dbxrefs=[]),
 ('C1',
  'TU1'): SeqRecord(seq=Seq('CATTACTCGCATCCATTCTCAGGCTGTCTCGTCTCGTCTCGGAGTTGGTCAGGG...TCG'), id='C1_TU1_PCR', name='C1_TU1_PCR', description='concatenated', dbxrefs=[]),
 ('C1',
  'TU2'): SeqRecord(seq=Seq('GCACTGAAGGTCCTCAATCGCACTGGAAACATCAAGGTCGGGAGTTGGTCAGGG...ACC'), id='C1_TU2_PCR', name='C1_TU2_PCR', description='concatenated', dbxrefs=[]),
 ('C2',
  'TU1'): SeqRecord(seq=Seq('CATTACTCGCATCCATTCTCAGGCTGTCTCGTCTCGTCTCGGAGTTGGTCAGGG...TCG'), id='C2_TU1_PCR', name='C2_TU1_PCR', description='concatenated', dbxrefs=[]),
 ('C

In [19]:
products[('C1','Backbone')]

SeqRecord(seq=Seq('CCAGGATACATAGATTACCACAACTCCGAGCCCTTCCACctactagtagcggcc...CTC'), id='C1_Backbone_PCR', name='C1_Backbone_PCR', description="PCR product with 5' overhangs (features preserved)", dbxrefs=[])

In [20]:
len(products[('C1','Backbone')])

5848